In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import numpy as np
import pyarrow.parquet as pq
import pandas as pd 
from pathlib import Path
from PIL import Image
from sklearn.preprocessing import StandardScaler

In [ ]:
PARQUET_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_train.parquet")
PHOTO_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/train/photos")
PARQUET_VAL_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_validate.parquet")
PHOTO_VAL_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/photos")

df = pq.read_table(source=PARQUET_PATH, use_threads=True).to_pandas()
df_val = pq.read_table(source=PARQUET_VAL_PATH, use_threads=True).to_pandas()

scaler = StandardScaler()
cols_to_scale = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
df_val[cols_to_scale] = scaler.fit_transform(df_val[cols_to_scale])

In [ ]:
class SimplePipeline:
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def process(self, image_path: Path):
        """Reads and transforms a single image for the NPU."""
        try:
            img = Image.open(image_path).convert('RGB')
            img_tensor = self.transform(img)
            return img_tensor.unsqueeze(0)
            
        except Exception as e:
            print(f"Failed to process {image_path.name}: {e}")
            return None

In [ ]:
class MyDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = Path(img_dir)
        self.transform = transform
        
        # Prepare tabular features for the MLP branch
        # We normalize these so the model doesn't choke on large numbers
        self.tabular_cols = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
        self.labels = self.df['growth'].values
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get the row data
        row = self.df.iloc[idx]
        
        # 2. Handle the Image
        # Assuming your file_name needs an extension like .jpg
        img_path = self.img_dir / f"{row['file_name']}.jpg"
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # 3. Handle the Tabular Data (MLP Input)
        # Convert the row features into a FloatTensor
        tab_data = torch.tensor(row[self.tabular_cols].values.astype('float32'))
        
        # 4. Handle the Label (Target)
        # Convert 'growth' (1, 2, 3) to a float for Regression
        label = torch.tensor([float(row['growth'])])
        
        return image, tab_data, label

In [ ]:
pipeline = MyPipeline(size=224)

train_ds = MyDataset(
    dataframe=df, 
    img_dir=PHOTO_PATH, 
    transform=pipeline.transform # Using data argumentation 
)

# 1. Define your counts from your previous log
counts = [124, 1039, 252] # Index 0=Stage 1, 1=Stage 2, 2=Stage 3
# Note: Adjust indices if your stages start at 1.0

# 2. Calculate weight for each class
class_weights = [1.0/c for c in counts]

# 3. Create a list of weights for EVERY sample in your dataset
# 'train_dataset.labels' should be a list of the stages for your training data
sample_weights = [class_weights[int(label) - 1] for label in train_ds.labels]

# 4. Create the Sampler
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)

val_ds = MyDataset(
    dataframe=df_val, 
    img_dir=PHOTO_VAL_PATH, 
    transform=pipeline.val_transform #use normal stuff
)

val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Simplified U-Net for DDPM: Inputs are (B, 3, 224, 224) noisy images + time embedding
        
        # Contracting Path (Encoder)
        self.enc1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2) # Downsample
        
        # Bottom bottleneck (optional: add attention layers here)
        self.bottleneck = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        
        # Expansive Path (Decoder with Skip Connections)
        self.dec1 = nn.ConvTranspose2d(128, 64, kernel_size=3, padding=1, output_padding=1, stride=2) # Upsample
        self.out = nn.Conv2d(64 + 64 + 64, 3, kernel_size=1)

        self.time_mlp = nn.Sequential(
            nn.Linear(1, 64),
            nn.ReLU(),
            nn.Linear(64, 64)
        )

    def forward(self, x, t):
        # 1. Process time into a feature vector
        t = t.float().unsqueeze(-1) / 1000.0
        t_feat = self.time_mlp(t) # Shape: [Batch, 64]
        
        # 2. Encoder Path
        e1 = self.enc1(x)  # e.g., [Batch, 64, 224, 224]
        e2 = self.enc2(e1) # e.g., [Batch, 128, 112, 112]
        
        # 3. Bottleneck
        b = self.bottleneck(e2)
        
        # 4. Decoder Path
        d1 = self.dec1(b)  # e.g., [Batch, 64, 224, 224]
        
        # --- THE FIX: BROADCASTING ---
        # We need to turn [Batch, 64] into [Batch, 64, Height, Width]
        # to match the spatial dimensions of our image features.
        t_spatial = t_feat.view(t_feat.shape[0], t_feat.shape[1], 1, 1)
        t_spatial = t_spatial.expand(-1, -1, d1.shape[2], d1.shape[3])
        
        # Now we can concatenate: (Decoder + Skip + Time)
        # Total channels = 64 (d1) + 64 (e1) + 64 (t_spatial) = 192
        combined = torch.cat((d1, e1, t_spatial), dim=1) 
        
        return self.out(combined)

In [ ]:
class SimpleDiffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="mps"):
        self.timesteps = timesteps
        self.device = device
        
        # Linear schedule for beta (noise intensity)
        self.betas = torch.linspace(beta_start, beta_end, timesteps).to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, axis=0)
        
    def add_noise(self, x_0, t):
        """The Forward Process: Clean -> Noisy"""
        noise = torch.randn_like(x_0)
        sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod[t])[:, None, None, None]
        sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod[t])[:, None, None, None]
        
        # Mix the image and the noise
        x_t = sqrt_alphas_cumprod * x_0 + sqrt_one_minus_alphas_cumprod * noise
        return x_t, noise

In [ ]:
device = torch.device("mps")
model = SimpleUNet().to(device) # Your UNet from before
diffusion = SimpleDiffusion(device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mse_loss = nn.MSELoss()

In [ ]:
for epoch in range(500): # Diffusion takes longer to converge than CNNs
    for images, _, _ in stage3_loader: # Only using Stage 3 images
        images = images.to(device)
        
        # 1. Sample random timesteps for the batch
        t = torch.randint(0, 1000, (images.shape[0],)).to(device)
        
        # 2. Add noise to the images
        x_noisy, true_noise = diffusion.add_noise(images, t)
        
        # 3. Predict the noise (The Reverse Process)
        # Note: Your UNet needs 't' as an input to know the noise level!
        predicted_noise = model(x_noisy, t) 
        
        # 4. Calculate Loss
        loss = mse_loss(predicted_noise, true_noise)
        
        # 5. Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()